In [29]:
import json
from typing import Any
import statistics
import random
from datetime import datetime, timezone

In [30]:
def generate_candles(start_time: int, count=1000, interval=300) -> list[dict]:
    """
    start_time: start in UNIX timestamp (ms)
    count: number of generated candles
    interval: (timeframe) in seconds
    """

    candles = []
    previous_close = 100
    previous_time = start_time

    for i in range(count):
        open_time = previous_time + i * interval * 1000

        open_price = previous_close
        close_price = round(open_price + random.gauss(0, 0.25), 2)

        high_price = round(max(open_price, close_price) + random.uniform(0.05, 0.3), 2)
        low_price = round(min(open_price, close_price) - random.uniform(0.05, 0.3), 2)

        candle = {
            "openTime": open_time,
            "closeTime": open_time + interval * 1000,
            "open": open_price,
            "high": high_price,
            "low": low_price,
            "close": close_price,
            "volume": round(random.uniform(1000, 4000), 1),
            "tradeNb": random.randint(100, 400),
        }

        candles.append(candle)
        previous_close = close_price

    return candles

In [31]:
def generate_moving_average(candles: list[dict], period=20) -> dict[str, Any]:
    values = []
    for i in range(period - 1, len(candles)):
        window = [candle["close"] for candle in candles[i - period + 1 : i + 1]]
        values.append({
            "timestamp": candles[i]["closeTime"],
            "value": round(sum(window) / period, 2),
        })

    return {
        "type": "line",
        "name": "Moving average",
        "color": "#4CAF50",
        "lineWidth": 2,
        "points": values,
    }

In [32]:
def generate_rsi(candles, period=14) -> dict[str, Any]:
    closes = [candle["close"] for candle in candles]
    points = []

    if len(closes) <= period:
        return {
            "type": "line",
            "name": "RSI",
            "color": "#9C27B0",
            "lineWidth": 2,
            "points": points,
        }

    changes = [closes[i] - closes[i - 1] for i in range(1, len(closes))]
    gains = [max(change, 0) for change in changes]
    losses = [max(-change, 0) for change in changes]

    avg_gain = sum(gains[:period]) / period
    avg_loss = sum(losses[:period]) / period

    for i in range(period, len(changes)):
        avg_gain = (avg_gain * (period - 1) + gains[i]) / period
        avg_loss = (avg_loss * (period - 1) + losses[i]) / period

        if avg_loss == 0:
            rsi = 100.0
        else:
            rs = avg_gain / avg_loss
            rsi = 100 - (100 / (1 + rs))

        points.append({
            "timestamp": candles[i + 1]["closeTime"],
            "value": round(rsi, 2),
        })

    # Add the first RSI point, calculated from the initial period.
    if avg_loss != 0:
        pass

    return {
        "type": "percent",
        "name": "RSI",
        "color": "#9C27B0",
        "lineWidth": 2,
        "points": points,
    }

In [33]:
def generate_bollinger_bands(candles, period=20, std_dev=2) -> dict[str, Any]:
    closes = [candle["close"] for candle in candles]
    points_low = []
    points_high = []

    for i in range(period - 1, len(closes)):
        window = closes[i - period + 1 : i + 1]
        average = statistics.mean(window)
        deviation = statistics.pstdev(window)

        timestamp = candles[i]["closeTime"]

        points_low.append({
            "timestamp": timestamp,
            "value": round(average - std_dev * deviation, 2),
        })
        points_high.append({
            "timestamp": timestamp,
            "value": round(average + std_dev * deviation, 2),
        })

    return {
        "type": "channel",
        "name": "Bollinger Bands",
        "color": "#FF9800",
        "lineWidth": 1,
        "pointsLow": points_low,
        "pointsHigh": points_high,
    }

In [34]:
import random


def generate_positions(candles, count, min_candle_wait=3):
    if len(candles) < 2 or count <= 0:
        return []

    candles = sorted(candles, key=lambda c: c["openTime"])
    positions = []
    i = 0

    while i < len(candles) - 1 and len(positions) < count:
        open_candle = candles[i]
        position_type = random.choice(["long", "short"])
        open_value = open_candle["close"]

        risk = open_value * random.uniform(0.005, 0.03)
        reward = risk * random.uniform(1.5, 3.0)

        if position_type == "long":
            stop_loss = open_value - risk
            take_profit = open_value + reward
        else:
            stop_loss = open_value + risk
            take_profit = open_value - reward

        close_idx = None
        close_value = None

        for j in range(i + 1, len(candles)):
            candle = candles[j]

            if position_type == "long":
                hit_sl = candle["low"] <= stop_loss
                hit_tp = candle["high"] >= take_profit
            else:
                hit_sl = candle["high"] >= stop_loss
                hit_tp = candle["low"] <= take_profit

            if hit_sl or hit_tp:
                # If both are hit in the same candle, assume SL first.
                close_value = stop_loss if hit_sl else take_profit
                close_idx = j
                break

        if close_idx is None:
            close_idx = len(candles) - 1
            close_value = candles[close_idx]["close"]

        close_candle = candles[close_idx]

        positions.append({
            "type": position_type,
            "openTime": open_candle["closeTime"],
            "openValue": open_value,
            "volume": round(random.uniform(0.1, 5.0), 4),
            "closeTime": close_candle["closeTime"],
            "closeValue": close_value,
            "stopLosses": [{
                "timestamp": open_candle["closeTime"],
                "value": round(stop_loss, 2),
            }],
            "takeProfits": [{
                "timestamp": open_candle["closeTime"],
                "value": round(take_profit, 2),
            }],
            "fees": round(random.uniform(0.1, 5.0), 2),
        })

        # Wait at least min_wait candles after closing.
        i = close_idx + min_candle_wait + 1

    return positions

In [35]:
timestamp = int(datetime(2020, 1, 1, tzinfo=timezone.utc).timestamp() * 1000)

candles = generate_candles(timestamp, 1000)

fileContent = {
    "candles": candles,
    "positions": generate_positions(candles, 5, 10),
    "indicators": [generate_bollinger_bands(candles), generate_rsi(candles), generate_moving_average(candles)],
}

with open("test-file.json", "w") as file:
    json.dump(fileContent, file, indent=2)